<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔧 Lab 03a: Add Function Tools to Your Travel Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 ここで学ぶこと

このラボでは、Contoso Travelのエージェントが**関数ツール**を使用して**実際のデータ**を検索できる機能を提供します。フライト、ホテル、レンタカーのCSVデータを検索するPython関数を定義し、エージェントが呼び出せるツールとして登録します。これにより、コンシェルジュは一般的なチャットボットからデータ駆動型の旅行アシスタントへと進化します。

> **ツールをスーパーパワーと考えてください。** ツールがなければ、エージェントは話すことしかできません。ツールがあれば、フライトを検索したり、ホテルの在庫を確認したり、レンタカーの空き状況を確認したりできます — これにより、チャットボットがデータ駆動型のアシスタントに変わります。

ツール呼び出しのループは次のように機能します:

```
User Query → Agent (LLM decides to call a tool)
                → function_call request
                    → Python function executes
                        → results returned to Agent
                            → Agent synthesizes final answer
```

---


## 🧳 Contoso Travelのストーリー

Lab 02のContoso Travelコンシェルジュは素晴らしい会話を持つことができます — 温かく、旅行に関する知識が豊富で、フォローアップの質問にも自然に対応します。しかし、最初の内部テストのラウンドで明らかになった重大な問題があります。テスターが「シアトルからパリまでのフライトを800ドル以下で探して」と尋ねたところ、コンシェルジュは自信を持ってフライトを推薦しましたが... **存在しないフライトでした。** 航空会社、価格、出発時間を幻覚しました。別のテスターが東京のホテルについて尋ねたところ、Contosoの在庫に存在しない物件を推薦されました。

### 🔴 現在直面している問題

- **エージェントが事実を捏造しています。** 実際のデータにアクセスできないため、もっともらしいが架空の情報で穴を埋めてしまいます。
- 旅行代理店にとって、これは致命的です — 顧客はこれらの推薦に基づいて予約を行います。
- エージェントがContosoの実際のフライト、ホテル、レンタカーの在庫を検索し、確認済みの結果のみを返す必要があります。
- しかし、AIエージェントを構造化データソースに接続するにはどうすればよいでしょうか？

### ✅ このラボで解決すること

このラボでは、**関数ツール**を紹介します — エージェントがPython関数を呼び出して実際のデータを照会できる仕組みです:
 - フライト、ホテル、レンタカーの検索関数を定義する、
 - エージェントが呼び出せるツールとして登録する、そして
 - 呼び出しと応答のループを処理する。

ラボの最後には、顧客がパリ行きのフライトについて尋ねたとき、エージェントは実際のCSV在庫を検索し、実際のオプションのみを返すようになります。

## 2. Setup

---


In [ ]:
# セットアップ: 呼び出し可能なツールを登録するための FunctionTool と、
# 基本型リストヒントとしての Tool をインポートします。
import os
import json
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, Tool, FunctionTool

# .env lives one level above the notebooks dir
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry")

## 3. 旅行データを読み込む

旅行CSVファイルをpandasのDataFrameに読み込みます。ツール関数はこれらのDataFrameを照会します。

---


In [ ]:
# パスはノートブックの場所に対して相対的です:
# notebooks/1-prompt-agents/ -> ../../data/\n
data_path = "../../data/contoso-travel"

flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")

print(f"✈️  {len(flights_df)} 件のフライトを読み込みました")
print(f"🏨 {len(hotels_df)} 件のホテルを読み込みました")
print(f"🚗 {len(cars_df)} 件のレンタカーを読み込みました")

## 4. ツール関数を定義する

各関数は、クエリパラメータに基づいて旅行データを検索し、一致する結果をJSON形式で返します。

---


In [ ]:
# Function toolsはJSON文字列を返す必要があります（辞書ではありません）。
# エージェントはツールの出力として生の文字列を受け取ります。

def search_flights(origin: str = None, destination: str = None, cabin_class: str = None, max_price: float = None) -> str:
    """指定された条件に基づいて利用可能なフライトを検索します。"""
    results = flights_df.copy()  # 元のデータを変更しないようにコピー
    if origin:
        # ユーザーの柔軟性のために大文字小文字を区別しない一致
        results = results[results["origin"].str.lower() == origin.lower()]
    if destination:
        results = results[results["destination"].str.lower() == destination.lower()]
    if cabin_class:
        results = results[results["cabin_class"].str.lower() == cabin_class.lower()]
    if max_price:
        results = results[results["price_usd"] <= max_price]
    
    if results.empty:
        return json.dumps({"message": "お客様の条件に一致するフライトは見つかりませんでした。", "results": []})
    
    # orient="records" -> list of dicts, easiest for LLM
    return results.to_json(orient="records")


def search_hotels(city: str = None, min_stars: int = None, max_price: float = None, amenities: str = None) -> str:
    """指定された条件に基づいて利用可能なホテルを検索します。"""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if min_stars:
        results = results[results["star_rating"] >= min_stars]
    if max_price:
        results = results[results["price_per_night_usd"] <= max_price]
    if amenities:
        # 部分一致: "Pool" は "Pool, Spa, WiFi" に一致します
        results = results[results["amenities"].str.lower().str.contains(amenities.lower())]
    
    if results.empty:
        return json.dumps({"message": "お客様の条件に一致するホテルは見つかりませんでした。", "results": []})
    
    return results.to_json(orient="records")


def search_car_rentals(city: str = None, car_type: str = None, max_price: float = None) -> str:
    """指定された条件に基づいて利用可能なレンタカーを検索します。"""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if car_type:
        results = results[results["car_type"].str.lower() == car_type.lower()]
    if max_price:
        results = results[results["price_per_day_usd"] <= max_price]
    # 常に利用可能なものだけをフィルタリング（フライトやホテルとは異なる）
    results = results[results["available"] == True]
    
    if results.empty:
        return json.dumps({"message": "お客様の条件に一致するレンタカーは見つかりませんでした。", "results": []})
    
    return results.to_json(orient="records")


# Quick test
print("🧪 search_flights('Seattle', 'Paris') のテスト:")
print(search_flights(origin="Seattle", destination="Paris"))

## 5. Function Tools の登録

各関数が受け入れるパラメータをエージェントに伝えるJSONスキーマを使用して、`FunctionTool` の定義を作成します。

---


In [ ]:
# FunctionTool: JSONスキーマは、各関数が受け入れるパラメータをエージェントに指示します。
# エージェントは、ツールを呼び出す際に、このスキーマに準拠した引数を生成します。

flight_tool = FunctionTool(
    name="search_flights",
    description="利用可能なフライトを検索します。顧客がフライト、航空運賃、目的地への飛行について尋ねたときに使用します。",
    parameters={
        "type": "object",
        "properties": {
            "origin": {"type": "string", "description": "出発都市名"},
            "destination": {"type": "string", "description": "到着都市名"},
            # enum constrains values the model can generate
            "cabin_class": {"type": "string", "description": "キャビンクラス: エコノミー、ビジネス、またはファースト", "enum": ["Economy", "Business", "First"]},
            "max_price": {"type": "number", "description": "最大チケット価格（USD）"},
        },
        "required": [],  # all params optional for flexible queries
        "additionalProperties": False,
    },
    strict=False,  # allows optional params (strict requires all)
)

hotel_tool = FunctionTool(
    name="search_hotels",
    description="利用可能なホテルを検索します。顧客がホテル、宿泊施設、滞在先について尋ねたときに使用します。",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "ホテルを検索する都市名"},
            "min_stars": {"type": "integer", "description": "最低星評価 (1-5)"},
            "max_price": {"type": "number", "description": "1泊あたりの最大価格（USD）"},
            "amenities": {"type": "string", "description": "フィルタリングする希望のアメニティ（例: プール、スパ、WiFi）"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

car_rental_tool = FunctionTool(
    name="search_car_rentals",
    description="利用可能なレンタカーを検索します。顧客がレンタカーや地上交通について尋ねたときに使用します。",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "レンタカーをピックアップする都市名"},
            "car_type": {"type": "string", "description": "車の種類: エコノミー、SUV、ラグジュアリー、またはミニバン", "enum": ["Economy", "SUV", "Luxury", "Minivan"]},
            "max_price": {"type": "number", "description": "1日あたりの最大価格（USD）"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

tools: list[Tool] = [flight_tool, hotel_tool, car_rental_tool]
print(f"✅ Defined {len(tools)} function tools: {[t.name for t in tools]}")

## 6. エンハンスメントしたトラベルエージェントを作成する

次に Function Tools を統合したエージェントを作成します。

---


In [ ]:
# トラベルエージェントのシステム指示を定義する
TRAVEL_AGENT_INSTRUCTIONS = """あなたは Contoso トラベルエージェントであり、リアルタイムの旅行データにアクセスできる専門の旅行アシスタントです。

以下のツールにアクセスできます:
- search_flights: 都市間の利用可能なフライトを検索します
- search_hotels: 目的地の都市でホテルを検索します  
- search_car_rentals: 都市でのレンタカーオプションを検索します

顧客が旅行オプションについて尋ねたとき:
1. 適切なツールを使用して在庫を検索します
2. 結果を明確で整理された形式で提示します
3. 価格、評価、空室状況などの関連情報を含めます
4. 結果に基づいて役立つ推奨事項を提供します

常に役立ち、正確で、専門的であること。結果が一致しない場合は、代替の基準を提案してください。"""

# tools= FunctionTool 定義をエージェントに添付することで、エージェントが
# 応答中に呼び出すことができる関数を認識します
agent = project_client.agents.create_version(
    agent_name="contoso-travel-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=TRAVEL_AGENT_INSTRUCTIONS,
        tools=tools,
    ),
)

print(f"✅ トラベルエージェントが {len(tools)} のツールで作成されました！")
print(f"   名前: {agent.name}")
print(f"   バージョン: {agent.version}")

## 7. テスト: フライト検索

フライト検索クエリでエージェントをテストします。エージェントは `search_flights` 関数ツールを呼び出します。

---


In [ ]:
# FunctionCallOutput: ツールの結果をモデルに返して合成を行う際に使用する型
from openai.types.responses.response_input_param import FunctionCallOutput

def handle_tool_calls(response):
    """Process any function_call output items and return results."""
    # ツール名をローカルのPython関数にマッピング
    tool_map = {
        "search_flights": search_flights,
        "search_hotels": search_hotels,
        "search_car_rentals": search_car_rentals,
    }
    
    function_results = []
    for item in response.output:
        # "function_call" items = エージェントがツールを要求している
        if item.type == "function_call":
            func = tool_map.get(item.name)
            if func:
                # 引数は JSON の文字列であり、dict ではありません。
                # arguments is a JSON string, not a dict
                args = json.loads(item.arguments)
                print(f"   🔧 Calling {item.name}({args})")
                result = func(**args)
                # call_id はこの結果をリクエストにリンクします
                function_results.append(
                    FunctionCallOutput(
                        type="function_call_output",
                        call_id=item.call_id,
                        output=result,
                    )
                )
    return function_results

# 新しい会話を開始してツール呼び出しをトリガーする
conversation = openai_client.conversations.create()

response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Find me flights from Seattle to Paris under $800.",
)

# レスポンスには関数呼び出し項目が含まれていますが、テキストはまだ含まれていません。
print("📨 エージェントがツール呼び出しを要求しました:")
function_results = handle_tool_calls(response)

## 8. Function Call のレスポンスの処理

エージェントが関数呼び出しを発行しました。関数を実行し、結果を返すことで、エージェントが最終的な応答を生成できるようにします。

---


In [ ]:
# ツールの結果をエージェントに送り返し、エージェントが生データから自然言語による回答を生成できるようにします。
# previous_response_id は、この処理を前のターンに関連付けます。
if function_results:
    response = openai_client.responses.create(
        input=function_results,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

print(f"\n🤖 トラベルエージェントの応答:\n")
print(response.output_text)

## 9. ホテル + 車のコンボのテスト

複数の tool call をトリガーする、より複雑なクエリをテストします。

---


In [ ]:
# Multi-tool: agent may call tools sequentially across
# rounds (hotel first, then car) rather than all at once
# Multi-tool：エージェントは、すべてのツールを一度に呼び出すのではなく、
# ラウンドごとに順番に呼び出す場合があります（Hotelが最初、次にCarなど）。

conversation2 = openai_client.conversations.create()

response = openai_client.responses.create(
    conversation=conversation2.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="東京で4つ星のホテルとレンタカーを探しています。ホテルの予算は1泊200ドルです。",
)

print("📨 エージェントがツール呼び出しをリクエストしました:")
function_results = handle_tool_calls(response)

if function_results:
    # 最初のバッチのツール結果を返します
    response = openai_client.responses.create(
        input=function_results,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

    # エージェントは2回目のツール呼び出しを行う場合があります
    # （例：最初にホテルを検索し、次に車を検索する場合）
    more_results = handle_tool_calls(response)
    if more_results:
        response = openai_client.responses.create(
            input=more_results,
            previous_response_id=response.id,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )

print(f"\n🤖 トラベルエージェントの応答:\n")
print(response.output_text)

## 10. Cleanup

---


In [ ]:
# Cleanup: delete both conversations and agent version
openai_client.conversations.delete(conversation_id=conversation.id)
openai_client.conversations.delete(conversation_id=conversation2.id)
print("✅ Conversations deleted")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ Agent version deleted")

openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed")

## 11. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ これまでの成果</h3>
  <ul style="margin-bottom: 0;">
  <li>CSVデータからフライト、ホテル、レンタカーを検索するPython関数を定義しました</li>
  <li>JSONスキーマを使用して<code>FunctionTool</code>を定義しました</li>
  <li>3つのツールすべてを組み込んだ強化されたトラベルエージェントを構築しました</li>
  <li>関数呼び出し → 実行 → 結果返却のワークフローを処理しました</li>
  <li>単一ツールおよび複数ツールのクエリをテストしました</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 重要なポイント:</strong> Function tools により、Contoso Travel エージェントは汎用チャットボットから、実際の在庫を照会できるデータ駆動型アシスタントに変わります。エージェントは顧客のクエリに基づいて、どのツールを呼び出すかを賢く判断します。
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ 次のステップ:</strong> <a href="lab-03b-workflow.ipynb">Lab 03b</a> では、専門化されたフライト、ホテル、レンタカーのエージェントをワークフロー定義でオーケストレーションする <b>マルチエージェントワークフロー</b> を作成します。
</div>

---
